In [16]:
"""
==========================================================================
Vermoeiingsbelasting windgolven — cyclustelling via Rayleigh-verdeling
==========================================================================
Bepaalt het aantal wisselingen (golven) en de bijbehorende golfhoogte-
verdeling over de levensduur van een constructieonderdeel.

Methodiek (per golfveld met één Hm0 en Tm):
  Stap 1  Binnen een golfveld zijn de individuele golfhoogtes Rayleigh
          verdeeld. Voor golfhoogtes van 5 cm t/m Hmax (stappen 5 cm)
          wordt met de Rayleigh-verdeling het tijdsaandeel per bin bepaald,
          en daaruit het aantal golven n (= aandeel * golven per tijdstap).
  Storm   35 uur totaal met 6 uur piek (volle Hs). De resterende 29 uur
          wordt gelijk verdeeld over opbouw en afbouw (14,5 uur elk),
          beide lineair. Elk tijdstap-blok is een eigen golfveld (eigen Hs).
  Stap 2  Aantal stormen per retourklasse over de levensduur.
  Stap 3  Sommeer over alle golfvelden en alle stormen -> N per golfhoogte
          (5 t/m 200 cm) over de totale levensduur.

Periodes:
  Invoer Tp = piekperiode. Voor de cyclustelling telt de gemiddelde
  (zero-crossing) periode Tm. JONSWAP-benadering:  Tm = Tp / 1.28.

Rayleigh (golfhoogtes):
  H_rms = Hm0 / sqrt(2)
  Overschrijdingskans:  P(H' > H) = exp( -(H / H_rms)^2 )
  Kans in bin [H1,H2]:  P = exp(-(H1/H_rms)^2) - exp(-(H2/H_rms)^2)
  Golven per uur:       N_uur = 3600 / Tm     [-]
==========================================================================
"""

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------
# 1) INVOER — stormklassen (uit Bijlage A/B, Hydra-NL, etc.)
# --------------------------------------------------------------------------
# n_storms = aantal stormen van deze klasse over de HELE levensduur.
#   1/1   per jaar  -> dagelijkse belasting        (freq * levensduur)
#   1/10  per jaar  -> jaarlijkse belasting
#   1/100 per jaar  -> 10-jaarlijkse belasting
#   1/1.000 & 1/10.000 per jaar -> "eens per levensduur"  -> n_storms = 1
#
# Tp = piekperiode [s]. Wordt intern omgezet naar Tm = Tp / TP_TO_TM.

LIFETIME_YEARS = 100.0          # ontwerplevensduur [jaar]
TP_TO_TM       = 1.28           # Tm = Tp / 1.28 (JONSWAP zero-crossing benadering)

storm_classes = [
    # label            Hs [m]    Tp [s]   n_storms over levensduur
    {"label": "1/1 per jaar",     "Hs": 1.33, "Tp": 8.15,  "n_storms": 1.0 * LIFETIME_YEARS},
    {"label": "1/10 per jaar",    "Hs": 1.72, "Tp": 8.65,  "n_storms": 0.1 * LIFETIME_YEARS},
    {"label": "1/100 per jaar",   "Hs": 2.19, "Tp": 10.11, "n_storms": 0.01 * LIFETIME_YEARS},
    {"label": "1/1.000 per jaar", "Hs": 2.64, "Tp": 11.01, "n_storms": 1.0},   # eens per levensduur
    {"label": "1/10.000 per jaar","Hs": 3.13, "Tp": 10.91, "n_storms": 1.0},   # eens per levensduur
]

# Golfhoogte-bins: 5 cm t/m 200 cm in stappen van 5 cm
DH   = 0.05                                   # binbreedte [m]
H_MIN, H_MAX = 0.05, 2.00                     # [m]
centers = np.arange(H_MIN, H_MAX + DH/2, DH)  # bin-middens [m]
edges_lo = centers - DH/2                      # ondergrens per bin [m]
edges_hi = centers + DH/2                      # bovengrens per bin [m]

# Storm: 35 uur totaal, 6 uur piek (volle Hs). Resterende 29 uur gelijk
# verdeeld over op- en afbouw (14,5 uur elk), beide lineair.
STORM_DURATION_H = 35.0                          # totale stormduur [uur]
PEAK_H           = 6.0                            # duur volle piek [uur]
RAMP_H           = (STORM_DURATION_H - PEAK_H) / 2.0   # op-/afbouw = 14,5 uur
DT               = 0.5                            # tijdstap [uur] (deelt 14,5 netjes)


# --------------------------------------------------------------------------
# 2) HULPFUNCTIES
# --------------------------------------------------------------------------
def rayleigh_prob_in_bins(Hs, lo, hi):
    """Kans dat een individuele golfhoogte in bin [lo, hi] valt (Rayleigh)."""
    H_rms = Hs / np.sqrt(2.0)
    P = np.exp(-(lo / H_rms) ** 2) - np.exp(-(hi / H_rms) ** 2)
    return np.clip(P, 0.0, None)


def storm_Hs_profile(Hs_peak):
    """
    Hs per tijdstap-blok voor de 35-uurs storm (6 u piek, 14,5 u lineaire
    op- en afbouw). Geeft (Hs_blok, uren_blok) terug per blok van DT uur.
    De Hs wordt op het midden van elk blok geevalueerd.
    """
    t_edges = np.arange(0.0, STORM_DURATION_H + DT/2, DT)
    t_mid   = (t_edges[:-1] + t_edges[1:]) / 2.0     # midden van elk blok [uur]

    t_peak_start = RAMP_H                              # 14,5 u
    t_peak_end   = RAMP_H + PEAK_H                     # 20,5 u

    Hs_blok = np.empty_like(t_mid)
    for i, t in enumerate(t_mid):
        if t < t_peak_start:                          # lineaire opbouw
            Hs_blok[i] = Hs_peak * (t / RAMP_H)
        elif t <= t_peak_end:                         # piek
            Hs_blok[i] = Hs_peak
        else:                                         # lineaire afbouw
            Hs_blok[i] = Hs_peak * (1.0 - (t - t_peak_end) / RAMP_H)

    return Hs_blok, np.full_like(t_mid, DT)           # (Hs per blok, uren per blok)


# --------------------------------------------------------------------------
# 3) CYCLUSTELLING — Stap 1 t/m 3
# --------------------------------------------------------------------------
total_cycles = np.zeros_like(centers)          # N per golfhoogte-bin over levensduur
overview_rows = []                             # invoer + afgeleide waarden per klasse

for storm in storm_classes:
    Hs_peak, Tp, n_storms = storm["Hs"], storm["Tp"], storm["n_storms"]
    Tm = Tp / TP_TO_TM                           # piek -> gemiddelde periode [s]
    overview_rows.append({
        "Klasse":      storm["label"],
        "Hs [m]":      Hs_peak,
        "Tp [s]":      Tp,
        "Tm [s]":      round(Tm, 2),
        "n_stormen":   n_storms,
    })
    N_per_hour = 3600.0 / Tm                     # aantal golven per uur [-]

    Hs_blocks, hrs_blocks = storm_Hs_profile(Hs_peak)
    cycles_per_storm = np.zeros_like(centers)
    for Hs_b, hrs_b in zip(Hs_blocks, hrs_blocks):   # per blok = 1 golfveld
        if Hs_b <= 0:
            continue
        N_block = N_per_hour * hrs_b             # aantal golven in dit blok [-]
        P_bins  = rayleigh_prob_in_bins(Hs_b, edges_lo, edges_hi)
        cycles_per_storm += P_bins * N_block     # n-getal per bin, dit blok

    total_cycles += cycles_per_storm * n_storms  # Stap 2 & 3: * aantal stormen


# --------------------------------------------------------------------------
# 4) RESULTAAT
# --------------------------------------------------------------------------
df = pd.DataFrame({
    "H [m]":        np.round(centers, 3),
    "N_cycli [-]":  total_cycles,
})
df["N_cycli [-]"] = df["N_cycli [-]"].round(0)

df_overview = pd.DataFrame(overview_rows)

print(f"Levensduur                : {LIFETIME_YEARS:.0f} jaar")
print(f"Stormduur                 : {STORM_DURATION_H:.0f} uur "
      f"({PEAK_H:.0f} u piek, 2x {RAMP_H:.1f} u op-/afbouw)")
print(f"Tp -> Tm factor           : / {TP_TO_TM}")
print(f"Totaal aantal golfcycli   : {total_cycles.sum():.3e}\n")

print("Ingevoerde stormklassen (invoer + afgeleide Tm):")
print(df_overview.to_string(index=False))
print()

print("Cyclustelling per golfhoogte over de levensduur:")
print(df.to_string(index=False))


# --------------------------------------------------------------------------
# 5) (OPTIONEEL) OMZETTEN NAAR SPANNINGSRANGE VOOR PALMGREN-MINER
# --------------------------------------------------------------------------
# Golfhoogte -> spanningsrange (top-dal) via je constructieve analyse:
#   delta_sigma = SIGMA_PER_M * H     [MPa]
#
# SIGMA_PER_M = 20.0                              # [MPa/m]
# df["dSigma [MPa]"] = SIGMA_PER_M * df["H [m]"]
# print(df.to_string(index=False))


Levensduur                : 100 jaar
Stormduur                 : 35 uur (6 u piek, 2x 14.5 u op-/afbouw)
Tp -> Tm factor           : / 1.28
Totaal aantal golfcycli   : 2.112e+06

Ingevoerde stormklassen (invoer + afgeleide Tm):
           Klasse  Hs [m]  Tp [s]  Tm [s]  n_stormen
     1/1 per jaar    1.33    8.15    6.37      100.0
    1/10 per jaar    1.72    8.65    6.76       10.0
   1/100 per jaar    2.19   10.11    7.90        1.0
 1/1.000 per jaar    2.64   11.01    8.60        1.0
1/10.000 per jaar    3.13   10.91    8.52        1.0

Cyclustelling per golfhoogte over de levensduur:
 H [m]  N_cycli [-]
  0.05     156449.0
  0.10     152056.0
  0.15     144276.0
  0.20     136571.0
  0.25     128973.0
  0.30     121476.0
  0.35     114103.0
  0.40     106874.0
  0.45      99809.0
  0.50      92929.0
  0.55      86251.0
  0.60      79794.0
  0.65      73574.0
  0.70      67606.0
  0.75      61905.0
  0.80      56481.0
  0.85      51343.0
  0.90      46499.0
  0.95      41952.0
  1.